In [30]:
import pandas as pd

df = pd.read_csv("data/orders.csv")
df

,order_id,city,distance_km,price,status,order_date
0,1,Dhaka,12,500,delivered,2024-01-01
1,2,Dhaka,25,800,cancelled,2024-01-01
2,3,Chattogram,40,1200,delivered,2024-01-02
3,4,Dhaka,18,600,delivered,2024-01-02
4,5,Sylhet,55,1500,pending,2024-01-03
5,6,Dhaka,30,1000,delivered,2024-01-03


In [31]:
import sqlite3

conn = sqlite3.connect("orders.db")


In [32]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   order_id     6 non-null      int64
 1   city         6 non-null      str  
 2   distance_km  6 non-null      int64
 3   price        6 non-null      int64
 4   status       6 non-null      str  
 5   order_date   6 non-null      str  
dtypes: int64(3), str(3)
memory usage: 420.0 bytes


,order_id,distance_km,price
count,6.000000,6.00000,6.000000
mean,3.500000,30.00000,933.333333
std,1.870829,15.60769,377.712413
min,1.000000,12.00000,500.000000
25%,2.250000,19.75000,650.000000
50%,3.500000,27.50000,900.000000
75%,4.750000,37.50000,1150.000000
max,6.000000,55.00000,1500.000000


In [33]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["status"] = df["status"].str.lower()

## ***Question 1: Total revenue***

In [34]:
df["price"].sum()

np.int64(5600)

## ***Question 2: Revenue by city***

In [35]:
df.groupby("city")["price"].sum()

city
Chattogram    1200
Dhaka         2900
Sylhet        1500
Name: price, dtype: int64

In [36]:
pd.read_sql("""
SELECT city, SUM(price) as total_revenue
FROM orders
GROUP BY city
""", conn)

,city,total_revenue
0,Chattogram,1200
1,Dhaka,2900
2,Sylhet,1500


## ***Question 3: Delivery success rate***

In [37]:
df["status"].value_counts(normalize=True) * 100

status
delivered    66.666667
cancelled    16.666667
pending      16.666667
Name: proportion, dtype: float64

In [39]:
pd.read_sql("""
SELECT status, COUNT(*) as count
FROM orders
GROUP BY status
""", conn)


,status,count
0,cancelled,1
1,delivered,4
2,pending,1


## ***Question 4: Average price by distance bucket***

In [38]:
df["distance_bucket"] = pd.cut(
    df["distance_km"], bins=[0,20,40,100]
)

df.groupby("distance_bucket")["price"].mean()

distance_bucket
(0, 20]       550.0
(20, 40]     1000.0
(40, 100]    1500.0
Name: price, dtype: float64